# MNIST demo with CREST


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Save the start time
import time
start_time = time.time()

# Allow crest to be imported
%cd -q ..

/Users/draghun1/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
from crest import DataServer

import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense, Lambda
from crest.model.HierarchalTensorGraph import HierarchalTensorGraph
from crest.model.KerasNode import KerasNode
from crest.model import TensorSpec


/Users/draghun1/miniconda3/envs/crest_cpu/lib/python3.10/site-packages/tensorflow/python/keras/engine/training_arrays_v1.py:37: UserWarning: A NumPy version >=1.22.4 and <1.29.0 is required for this version of SciPy (detected version 1.22.3)
  from scipy.sparse import issparse  # pylint: disable=g-import-not-at-top


#### Load MNIST Datasets


In [4]:
# Load datasets from server
ds_train, ds_valid, ds_test, ds_pred = DataServer.load('mnist')

# Get input/output labels
[INP, OUT] = ds_train.labels

# Num classes
n_classes = 10
patch_size = 28

#### Create the HierarchalTensorGraph


In [5]:

layers = [
    KerasNode(Lambda(lambda x: tf.cast(x, tf.float32)), name='layer_0', inputs={INP: TensorSpec(
        [None, patch_size, patch_size])}, outputs={OUT: TensorSpec([None, 28, 28])}),        # Cast dtype
    KerasNode(Lambda(lambda x: tf.expand_dims(x, axis=-1)), name='layer_1', inputs={INP: TensorSpec(
        [None, patch_size, patch_size])}, outputs={OUT: TensorSpec([None, 28, 28, 1])}),  # Add channel dimension
    KerasNode(Conv2D(10, 3, padding='same', activation='relu'), name='layer_2', inputs={INP: TensorSpec(
        [None, 28, 28, 1])}, outputs={OUT: TensorSpec([None, 28, 28, 10])}),  # Convolutional layer
    KerasNode(Flatten(), name='layer_3', inputs={INP: TensorSpec(
        [None, 28, 28, 10])}, outputs={OUT: TensorSpec([None, 28*28*10])}),  # Flatten
    KerasNode(Dropout(0.3), name='layer_4', inputs={INP: TensorSpec(
        [None, 28*28*10])}, outputs={OUT: TensorSpec([None, 28*28*10])}),  # Dropout
    KerasNode(Dense(n_classes, activation='softmax'), name='layer_5', inputs={
              INP: TensorSpec([None, 28*28*10])}, outputs={OUT: TensorSpec([None, 10])})  # Output layer
]

for i in layers[:-1]:
    i.rename_io(outputs_map={OUT: INP})

graph = HierarchalTensorGraph(
    name='MNIST',
    inputs={INP: TensorSpec([None, patch_size, patch_size])},
    outputs={OUT: TensorSpec([None, 10])}
)

graph.add_edge('input', layers[0])

for i, layer in enumerate(layers[1:]):
    graph.add_edge(layers[i], layers[i+1])

graph.add_edge(layers[-1], 'output')

#### Build a model using the HTG


In [6]:
from crest import Model

# Define a custom metric if desired
def custom_metric(y_true, y_pred):
    squared_difference = tf.square(y_true - y_pred)
    return tf.reduce_mean(squared_difference, axis=-1)


# Set build options
build_kwargs = {
    'optimizer': 'Adam',
    'loss': 'sparse_categorical_crossentropy',
    'metrics': ['sparse_categorical_accuracy', custom_metric]
}

# Create anad build a model using the HTG
model = Model(graph)
model.build(**build_kwargs)

/Users/draghun1/Documents/Code/crest/crest/model/Model.py:89: UserWarning: Use Model.compile instead of Model.build
  warnings.warn('Use Model.compile instead of Model.build')


#### Train the model


In [7]:
# Create the training and validation batchers
from crest.data.Batcher import Batcher

batch_kwargs = {
    'batch_size': 200,            # 200 samples in each batch
    'features': [[INP], [OUT]],  # [{'image': ...}, {'class': ...}]
    'repeat': True,           # Yield indefinitely
    'workers': 0               # Number of background processes
}

batch_train = Batcher(ds_train, **batch_kwargs)
batch_kwargs['shuffle'] = False    # Turn shuffle off for prediction
batch_valid = Batcher(ds_valid,  **batch_kwargs)

with batch_train as train, batch_valid as valid:
    model.fit(train, **{
        'validation_data': valid,
        'steps_per_epoch': 250,  # 50k samples / 200 samples per batch
        'validation_steps': 50,  # 10k samples / 200 samples per batch
        'epochs': 5,
        'verbose': 1
    })

Epoch 1/5


  1/250 [..............................] - ETA: 38s - loss: 60.1300 - sparse_categorical_accuracy: 0.1050 - custom_metric: 27.9828

  9/250 [>.............................] - ETA: 2s - loss: 27.4788 - sparse_categorical_accuracy: 0.4167 - custom_metric: 28.3226 

 17/250 [=>............................] - ETA: 1s - loss: 18.6646 - sparse_categorical_accuracy: 0.5659 - custom_metric: 28.5919

 26/250 [==>...........................] - ETA: 1s - loss: 14.2558 - sparse_categorical_accuracy: 0.6519 - custom_metric: 27.9438

...

226/250 [==========================>...] - ETA: 0s - loss: 0.1366 - sparse_categorical_accuracy: 0.9688 - custom_metric: 27.4306

232/250 [==========================>...] - ETA: 0s - loss: 0.1351 - sparse_categorical_accuracy: 0.9689 - custom_metric: 27.4384

241/250 [===========================>..] - ETA: 0s - loss: 0.1342 - sparse_categorical_accuracy: 0.9689 - custom_metric: 27.4369

249/250 [============================>.] - ETA: 0s - loss: 0.1328 - sparse_categorical_accuracy: 0.9689 - custom_metric: 27.3968

250/250 [==============================] - 5s 18ms/step - loss: 0.1328 - sparse_categorical_accuracy: 0.9689 - custom_metric: 27.4044 - val_loss: 0.1901 - val_sparse_categorical_accuracy: 0.9740 - val_custom_metric: 27.4864


#### Evaluate the model


In [8]:
# Evaluate the model on the test set
batch_test = Batcher(ds_test, **batch_kwargs)
evaluate_kwargs = {
    'verbose': 1,
    'return_dict': True
}

with batch_test as batcher:
    res = model.evaluate(batcher, verbose=1, return_dict=1)

1/1 [==============================] - ETA: 0s - loss: 0.0179 - sparse_categorical_accuracy: 0.9950 - custom_metric: 26.3263

1/1 [==============================] - 0s 10ms/step - loss: 0.0179 - sparse_categorical_accuracy: 0.9950 - custom_metric: 26.3263


Accuracy on the full test set should be > ~97%.


### Make predictions


In [9]:
# Create batcher for predictions
batch_pred = Batcher(ds_pred, **{  # Reset batch iterator
    'batch_size': 200,
    'features': [INP],
    'repeat': True,
    'workers': 0,
    'shuffle': False
})

# Set kwargs for Model.predict() (can be the same as Keras.predict)
pred_kwargs = {
    'batch_size': 200,
    'verbose': False,
    'steps': 10000//200
}

# Generate predictions
with batch_pred as batcher:
    model_pred_batch = model.predict(
        batch_pred, **pred_kwargs)  # Using a batcher
    model_pred_dataset = model.predict(
        ds_pred, **pred_kwargs)   # Using a dataset
    print('Difference:',
          (model_pred_batch['class'] - model_pred_dataset['class']).sum())

Difference: 0.0


#### Adding additional output (e.g., lat and lon) when making a predictions


In [10]:
# Create inputs with additional features lat and lon
import numpy as np
from crest.data.loading import StructuredDataset
x = ds_pred.data[0]
lat = np.arange(len(x), dtype=float)
lon = np.arange(len(x), dtype=float)
x_aux = (lat, lon, x)
ds_aux = StructuredDataset(*x_aux, labels=['lat', 'lon', INP])

# Specify additional output as coords when making predictions
model.predict(ds_aux, coords=['lat', 'lon'], **pred_kwargs)

{'class': array([[3.79915654e-20, 9.36676788e-21, 7.72778386e-16, ...,
         1.00000000e+00, 8.18854750e-13, 2.96888030e-14],
        [2.10288231e-10, 5.17912434e-14, 9.99982119e-01, ...,
         1.38305589e-24, 5.40307436e-13, 1.04606389e-21],
        [1.14273826e-06, 9.96657133e-01, 1.42228755e-05, ...,
         4.54818235e-07, 1.32914778e-04, 6.35052499e-09],
        ...,
        [5.91514723e-20, 4.28995167e-15, 3.55744309e-16, ...,
         2.31790391e-08, 2.57248972e-10, 3.08407891e-07],
        [1.04514390e-16, 1.74754868e-19, 2.67676628e-19, ...,
         4.17252752e-19, 1.51114889e-08, 1.89885216e-17],
        [3.47087301e-18, 1.00385462e-26, 3.27781101e-11, ...,
         9.03922746e-20, 5.46244972e-18, 5.34934032e-19]], dtype=float32),
 'lat': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03]),
 'lon': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03])}

In [11]:
print('Elapsed time = ', time.time() - start_time, '[s]')

Elapsed time =  34.89463210105896 [s]


model save and load

In [12]:
model.save()

In [13]:
loaded_model = model.load('crest_cache')

In [14]:
# check that loaded model performs the same as original
result_loaded = loaded_model.predict(ds_aux, coords=['lat', 'lon'], **pred_kwargs)
result = model.predict(ds_aux, coords=['lat', 'lon'], **pred_kwargs)

print('Difference:', (result['class'] - result_loaded['class']).sum())

Difference: 0.0
